# Additive Correction Full Matrix (Result Display)

**Read-only display** of artifacts persisted under `results/exp_r12/` (generated by `023_exp_r12_additive_full_matrix.py`; this notebook performs no computation of its own).

## Background and Design

- **Motivation**: the paper only demonstrated additive correction on the GNN backbone (mechanism-isolation experiment, `tab:mechanism_isolation`). The "2 correction forms x 3 backbones x 3 signals" matrix therefore has gaps -- the six static-backbone arms (Uniform, GPM) x additive x {N, P, NP} were never reported, so readers could not tell whether "additive removes antagonism" is GNN-specific or general.
- **Full matrix** = 21 arms: 3 backbones (Uniform / GPM / GNN) x 7 variants (uncorrected, multiplicative N/P/NP, additive N/P/NP) x 16 regions x 3 metrics. GNN arms follow the established convention of computing per seed first and then averaging across seeds (the per-seed version is stored separately as `full_matrix_per_seed.csv`).
- **The correction definition is identical to the one used elsewhere** (shared module `shared_correction_utils`): additive = `base + alpha * (factor - factor_mean)`, followed by per-ITL3 renormalization, with alpha = base_std/offset_std (ddof=0).
- **Align before extending**: before adding the new arms, the existing 15 arms were validated against three independent anchors (stored-precision anchor from the frozen artifact / an independently recomputed full-precision anchor / a +-0.005 weak anchor against the paper's tables, plus a same-source strong anchor for the GNN multiplicative arms against the on-disk correction stored in the pickle) -- see `anchor_report.json`, bound by pytest `tests/test_r12.py`.
- **The Germany arm has been removed** (artifacts are missing from both local and backup storage).

## Structural Degeneration of Uniform x additive (Not a Bug)

The scale of the additive correction, alpha = base_std/offset_std, is supplied by the **backbone's own variance**. Within the Uniform backbone's regions, base_std = 0, so alpha = 0 and the correction is **exactly equal** to the baseline. The matrix reports this faithfully (all three arms match Uniform value-for-value), and `anchor_report.json` flags `structural_degeneration=true`. This follows directly from the earlier finding that "the additive scale is supplied by backbone variance": **the additive form cannot act on a zero-variance backbone** -- useful material for the methods section.

In [ ]:
# Read-only loading of artifacts
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

OUT = Path('..') / 'results' / 'exp_r12'
matrix = pd.read_csv(OUT / 'full_matrix.csv')
per_seed = pd.read_csv(OUT / 'full_matrix_per_seed.csv')
comps = pd.read_csv(OUT / 'new_comparisons.csv')
with open(OUT / 'anchor_report.json', encoding='utf-8') as f:
    report = json.load(f)

REGIONS = report['meta']['region_order']
print(f"full_matrix: {len(matrix)} rows (21 arms x 3 metrics); per_seed: {len(per_seed)} rows")
print(f"Anchoring: {report['anchor_summary']['n_checks']} checks, {report['anchor_summary']['n_failed']} failed")
for cat, entries in report['anchors'].items():
    if cat == 'not_anchored':
        for e in entries:
            print(f"  [unanchored, logged] {e['arm']} ({e['anchor']}): {e['reason']}")
    else:
        n_pass = sum(1 for v in entries.values() if v['passed'])
        print(f"  {cat}: {n_pass}/{len(entries)} passed")

## 1. Full-Matrix Heatmap (3 Backbones x 7 Variants, Mean over 16 Regions)

Lower RMSE / MAE is better; higher Corr is better. The three UniAdd cells are identical to Uni (structural degeneration, marked with `≡` in the cell).

In [ ]:
ARM_GRID = [
    ['Uni', 'UniN', 'UniP', 'UniNP', 'UniAddN', 'UniAddP', 'UniAddNP'],
    ['GPM', 'GPMpostN', 'GPMpostP', 'GPMpostNP', 'GPMaddN', 'GPMaddP', 'GPMaddNP'],
    ['GNN', 'GNNpostN', 'GNNpostP', 'GNNpostNP', 'GNNaddN', 'GNNaddP', 'GNNaddNP'],
]
BASE_NAMES = ['Uniform', 'GPM', 'GNN']
VARIANT_NAMES = ['Uncorrected', 'Mult N', 'Mult P', 'Mult NP', 'add N', 'add P', 'add NP']
DEGEN = {'UniAddN', 'UniAddP', 'UniAddNP'}

mean_of = {(r['arm'], r['metric']): r['mean_16regions'] for _, r in matrix.iterrows()}
std_of = {(r['arm'], r['metric']): r['std_ddof0_16regions'] for _, r in matrix.iterrows()}

fig, axes = plt.subplots(1, 3, figsize=(19, 4.6))
for ax, metric, title in zip(axes, ['rmse', 'mae', 'corr'],
                             ['RMSE (lower is better)', 'MAE (lower is better)', 'Corr (higher is better)']):
    M = np.array([[mean_of[(a, metric)] for a in row] for row in ARM_GRID])
    # Color direction: for error metrics, low = good = green; for corr, high = good = green
    cmap = 'RdYlGn_r' if metric != 'corr' else 'RdYlGn'
    im = ax.imshow(M, cmap=cmap, aspect='auto')
    for i in range(3):
        for j in range(7):
            tag = ' ≡' if ARM_GRID[i][j] in DEGEN else ''
            fmt = f'{M[i, j]:.2f}' if metric != 'corr' else f'{M[i, j]:.3f}'
            ax.text(j, i, fmt + tag, ha='center', va='center', fontsize=9)
    ax.set_xticks(range(7), VARIANT_NAMES, rotation=30, ha='right')
    ax.set_yticks(range(3), BASE_NAMES)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle('Full Matrix: 3 Backbones x 7 Variants (Mean over 16 Regions; ≡ = Structural Degeneration to Baseline)', y=1.04)
plt.tight_layout()
plt.show()

## 2. Key New Cross-Comparisons (Statistical Protocol)

seed average -> paired difference across 16 regions -> **exact sign-flip permutation** (exhaustive 2^16) + Holm correction (one family per metric, main family = 14 genuine comparisons) + region-level paired bootstrap CI (outer layer only, B=10^4). Static-vs-static comparisons use the no_seed branch. Negative delta means the first arm (additive) is better (RMSE/MAE); the direction is reversed for Corr.

**Headline comparison**: GNNaddP vs GPMpostNP -- the best additive-GNN arm against the globally best static arm.

In [ ]:
hl = report['headline_numbers']['GNNaddP_vs_GPMpostNP']
print('Headline: GNNaddP vs GPMpostNP')
for m in ['rmse', 'mae', 'corr']:
    print(f"  {m:5s}: Δ = {hl[m]['mean_diff']:+.4f}, perm p = {hl[m]['perm_p']:.4g}, "
          f"Holm p = {hl[m]['holm_p']:.4g}")

cols = ['comparison_id', 'comparison', 'metric', 'seed_branch', 'mean_diff',
        'perm_p', 'holm_p', 'new_sig', 'ci_lo', 'ci_hi']
main = comps[comps['family'] == 'main'][cols].copy()
for c in ['mean_diff', 'ci_lo', 'ci_hi']:
    main[c] = main[c].map(lambda v: f'{v:+.4f}')
for c in ['perm_p', 'holm_p']:
    main[c] = main[c].map(lambda v: f'{v:.3g}')
main.sort_values(['metric', 'comparison_id'])

## 3. additive-static vs multiplicative-static

The core question behind the earlier matrix gap: does additive perform as well (or worse) as multiplicative on **static backbones**? The GPM backbone gives a non-degenerate answer; on the Uniform backbone, additive fails structurally (see Section 4).

In [ ]:
rows = []
for sig in ['N', 'P', 'NP']:
    for m in ['rmse', 'mae', 'corr']:
        add_m = mean_of[(f'GPMadd{sig}', m)]
        mult_m = mean_of[(f'GPMpost{sig}', m)]
        hit = comps[(comps['arm_a'] == f'GPMadd{sig}')
                    & (comps['arm_b'] == f'GPMpost{sig}') & (comps['metric'] == m)]
        rows.append({'signal': sig, 'metric': m,
                     'GPM additive': round(add_m, 3), 'GPM mult': round(mult_m, 3),
                     'Δ(add−mult)': round(add_m - mult_m, 3),
                     'perm p': float(hit['perm_p'].iloc[0]),
                     'Holm p': float(hit['holm_p'].iloc[0]),
                     'significant': bool(hit['new_sig'].iloc[0])})
pd.DataFrame(rows)

In [ ]:
# The same question on the GNN backbone (a formal test of the paper's existing finding) + additive's gain over baseline
rows = []
for base in ['GNN']:
    for sig in ['N', 'P', 'NP']:
        for m in ['rmse']:
            add_m = mean_of[(f'{base}add{sig}', m)]
            mult_m = mean_of[(f'{base}post{sig}', m)]
            base_m = mean_of[(base, m)]
            rows.append({'signal': sig, 'GNN baseline': round(base_m, 3),
                         'GNN additive': round(add_m, 3), 'GNN mult': round(mult_m, 3),
                         'Δ(add−baseline)': round(add_m - base_m, 3),
                         'Δ(add−mult)': round(add_m - mult_m, 3)})
print('GNN backbone (RMSE, mean over 16 regions) -- additive uniformly outperforms multiplicative and the baseline:')
pd.DataFrame(rows)

## 4. Verification of Uniform x additive Structural Degeneration

- **Premise**: within each Uniform-backbone ITL3 group, values are exactly identical (ptp = 0, verified on disk) -> `base_std = 0` -> alpha = base_std/offset_std = 0 -> the additive correction is identically equal to the baseline.
- **Verification**: the UniAddN / UniAddP / UniAddNP arms match the Uni baseline value-for-value across all 16 regions x 3 metrics (tolerance 1e-9, with only residual floating-point noise from the idempotent renormalization); bound by pytest.
- **Argumentative value**: this is not an implementation defect but a direct consequence of the earlier finding that "the additive scale is supplied by backbone variance" -- the additive correction **automatically fails** on a zero-variance backbone; its precondition for effectiveness is that the backbone supplies spatial variance. By contrast, multiplicative correction remains effective on the Uniform backbone (UniN/UniP/UniNP all improve), because the multiplicative factor carries its own scale.

In [ ]:
deg = report['uniform_additive_degeneration']
print(f"structural_degeneration = {deg['structural_degeneration']}")
print(f"Max within-group ptp = {deg['uniform_base_within_group_ptp_max']:.3e} (exact-identity premise)")
print(f"Max within-group numpy std = {deg['uniform_base_within_group_std_max']:.3e} (floating-point mean noise)")
print(f"Max agent-level demand deviation: " + ', '.join(
    f"{k}={v:.3e}" for k, v in deg['max_agent_demand_abs_dev_by_signal'].items()))
print(f"Max metric-level deviation (UniAdd vs Uni) = "
      f"{max(deg['metric_level_max_abs_dev'].values()):.3e} (tolerance {deg['tol']:.0e})")
print(f"All within tolerance: {deg['all_within_tol']}")

deg_rows = comps[comps['family'] == 'degeneration']
deg_rows[['comparison', 'metric', 'mean_diff', 'perm_p', 'seed_branch']]

## 5. Per-Seed Transparency (Comparisons Involving GNN)

The permutation p-values from all 3 seeds are reported in full without merging (this directly addresses the earlier concern about using a median p-value); the Cauchy-combined column is for reference only, not the primary statistic.

In [ ]:
gnn_comps = comps[comps['seed_branch'] == 'seed_averaged']
cols = ['comparison', 'metric', 'perm_p',
        'perm_p_seed_42', 'perm_p_seed_123', 'perm_p_seed_456', 'cauchy_combined_p']
tbl = gnn_comps[cols].copy()
for c in cols[2:]:
    tbl[c] = tbl[c].map(lambda v: f'{v:.3g}')
tbl


## 6. Key Conclusions (All Numbers Traced to On-Disk Artifacts)

1. **Full-matrix completion**: the 6 previously missing arms (static backbones x additive) are now persisted; the GPM backbone gives non-degenerate additive results, while the Uniform backbone degenerates structurally (reported and flagged as such).
2. **Headline comparison** (see the printout in Section 2): the delta and permutation p-value for GNNaddP vs GPMpostNP.
3. **Backbone dependence of additive vs multiplicative**: on the GNN backbone, additive is significantly better than multiplicative (antagonism is removed); the relative performance of the two on the GPM backbone is shown in the Section 3 table -- this directly answers whether additive's advantage is GNN-specific.
4. All new comparisons follow the established statistical protocol (exact 2^16 permutation + one Holm family per metric + outer-layer bootstrap CI), with protocol fields consistent with the earlier on-disk artifacts (bound by pytest).